# Geolocation Data Cleaning

Problems to investigate:
- 261831 Duplicate rows 
- Check longitude and latitude ranges
- Any whitespace

In [1]:
import pandas as pd 

geoloc_raw = pd.read_csv('../data/raw/olist_geolocation_dataset.csv')
geoloc = geoloc_raw.copy()

print("Original Shape:", geoloc_raw.shape)
print("Working Shape:", geoloc.shape)

Original Shape: (1000163, 5)
Working Shape: (1000163, 5)


In [2]:
# Investigate duplicate rows

print("Duplicate rows:", geoloc.duplicated().sum())

# Identify all rows involved in duplicates
duplicate_geolocation = geoloc[
    geoloc.duplicated(keep=False)
]

print("Rows involved in duplicates:", len(duplicate_geolocation))

display(duplicate_geolocation.head(10))

Duplicate rows: 261831
Rows involved in duplicates: 390005


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.545621,-46.639292,sao paulo,SP
1,1046,-23.546081,-46.644820,sao paulo,SP
2,1046,-23.546129,-46.642951,sao paulo,SP
6,1047,-23.546273,-46.641225,sao paulo,SP
7,1013,-23.546923,-46.634264,sao paulo,SP
8,1029,-23.543769,-46.634278,sao paulo,SP
9,1011,-23.547640,-46.636032,sao paulo,SP
10,1013,-23.547325,-46.634184,sao paulo,SP
13,1012,-23.548946,-46.634671,sao paulo,SP
15,1046,-23.546081,-46.644820,sao paulo,SP


In [13]:
# Check duplicate latitude/longitude combinations
print(
    "Duplicate coordinate occurrences:",
    geoloc.duplicated(
        subset=["geolocation_lat", "geolocation_lng"]
    ).sum()
)

duplicate_coordinates = geoloc[
    geoloc.duplicated(
        subset=["geolocation_lat", "geolocation_lng"],
        keep=False
    )
].sort_values(
    ["geolocation_lat", "geolocation_lng"]
)

display(duplicate_coordinates.head(20))

Duplicate coordinate occurrences: 281700


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
993075,98780,-36.603837,-64.287433,santa rosa,RS
993302,98780,-36.603837,-64.287433,santa rosa,RS
978860,96255,-33.692136,-53.463279,chui,RS
979282,96255,-33.692136,-53.463279,chuí,RS
978479,96255,-33.691673,-53.450672,chui,RS
978749,96255,-33.691673,-53.450672,chui,RS
979087,96255,-33.691550,-53.454576,chuí,RS
980017,96255,-33.691550,-53.454576,chuí,RS
978429,96255,-33.691424,-53.480155,chui,RS
978678,96255,-33.691424,-53.480155,chui,RS


In [14]:
# Remove exact duplicate rows
geoloc = geoloc.drop_duplicates().copy()

print("Dataset shape after removing duplicates:", geoloc.shape)
print("Remaining exact duplicates:", geoloc.duplicated().sum())

Dataset shape after removing duplicates: (738332, 5)
Remaining exact duplicates: 0


In [15]:
# Investigate city spelling variations
import unicodedata

def normalize_city(city):
    return "".join(
        character for character in unicodedata.normalize("NFD", city)
        if unicodedata.category(character) != "Mn"
    )

geoloc["city_normalized"] = (
    geoloc["geolocation_city"].str.lower().apply(normalize_city)
)

city_variations = (
    geoloc.groupby("city_normalized")["geolocation_city"]
    .nunique()
)

print("Potential city spelling variations:", (city_variations > 1).sum())

Potential city spelling variations: 2036


In [17]:
# Display city names with potential spelling variations
variation_groups = city_variations[city_variations > 1]

display(
    geoloc[
        geoloc["city_normalized"].isin(variation_groups.index)
    ][["geolocation_city", "city_normalized", "geolocation_state"]]
    .drop_duplicates()
    .sort_values("city_normalized")
    .head(50)
)

,geolocation_city,city_normalized,geolocation_state
792364,abadiania,abadiania,GO
792411,abadiânia,abadiania,GO
598338,abaeté,abaete,MG
598309,abaete,abaete,MG
697155,abaira,abaira,BA
697165,abaíra,abaira,BA
702417,abaré,abare,BA
702410,abare,abare,BA
888351,abatia,abatia,PR
888733,abatiá,abatia,PR


In [30]:
duplicate_locations = geoloc[
    geoloc.duplicated(
        subset=[
            "geolocation_zip_code_prefix",
            "geolocation_lat",
            "geolocation_lng",
            "city_normalized",
            "geolocation_state"
        ],
        keep=False
    )
].sort_values(
    ["geolocation_zip_code_prefix", "city_normalized"]
)

display(duplicate_locations.head(20))

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,city_normalized,geolocation_city_normalized


In [27]:
geoloc = geoloc.drop_duplicates(
    subset=[
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng",
        "city_normalized",
        "geolocation_state"
    ]
).copy()

In [28]:
print("Duplicate rows:", geoloc.duplicated().sum())

Duplicate rows: 0


In [23]:
print("Remaining duplicate geographic mappings:",
      geoloc.duplicated(
          subset=[
              "geolocation_zip_code_prefix",
              "geolocation_lat",
              "geolocation_lng",
              "city_normalized",
              "geolocation_state"
          ]
      ).sum())

print("Final dataset shape:", geoloc.shape)

Remaining duplicate geographic mappings: 0
Final dataset shape: (720496, 7)


In [32]:
duplicate_location_groups = (
    geoloc.groupby(
        [
            "city_normalized",
            "geolocation_zip_code_prefix",
            "geolocation_lat",
            "geolocation_lng"
        ]
    )
    .size()
    .reset_index(name="record_count")
    .query("record_count > 1")
    .sort_values(
        ["city_normalized", "geolocation_zip_code_prefix"]
    )
)

print("Duplicate location groups:", len(duplicate_location_groups))
display(duplicate_location_groups)

Duplicate location groups: 2


,city_normalized,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,record_count
480580,rio de janeiro,23056,-22.919164,-43.611097,2
600349,sao paulo,4011,-23.578707,-46.645779,2


In [33]:
duplicate_locations = geoloc.merge(
    duplicate_location_groups[
        [
            "city_normalized",
            "geolocation_zip_code_prefix",
            "geolocation_lat",
            "geolocation_lng"
        ]
    ],
    on=[
        "city_normalized",
        "geolocation_zip_code_prefix",
        "geolocation_lat",
        "geolocation_lng"
    ],
    how="inner"
).sort_values(
    ["city_normalized", "geolocation_zip_code_prefix"]
)

display(duplicate_locations)

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,city_normalized
2,23056,-22.919164,-43.611097,rio de janeiro,AC,rio de janeiro
3,23056,-22.919164,-43.611097,rio de janeiro,RJ,rio de janeiro
0,4011,-23.578707,-46.645779,são paulo,SP,sao paulo
1,4011,-23.578707,-46.645779,sao paulo,AC,sao paulo


In [36]:
# Final check for exact duplicate rows
print("Exact duplicate rows:", geoloc.duplicated().sum())

# Final check for duplicate location combinations
location_columns = [
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng",
    "city_normalized",
    "geolocation_state"
]

duplicate_locations = geoloc[
    geoloc.duplicated(subset=location_columns, keep=False)
].sort_values(location_columns)

print("Rows with duplicate location combinations:", len(duplicate_locations))

display(duplicate_locations)

Exact duplicate rows: 0
Rows with duplicate location combinations: 0


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,city_normalized


In [3]:
# Check coordinate ranges
print("Latitude range:",
      geoloc["geolocation_lat"].min(),
      "to",
      geoloc["geolocation_lat"].max())

print("Longitude range:",
      geoloc["geolocation_lng"].min(),
      "to",
      geoloc["geolocation_lng"].max())

Latitude range: -36.6053744107061 to 45.06593318269697
Longitude range: -101.46676644931476 to 121.10539381057764


In [4]:
# Find potentially invalid coordinates
invalid_coordinates = geoloc[
    ~geoloc["geolocation_lat"].between(-34, 6) |
    ~geoloc["geolocation_lng"].between(-74, -34)
]

print("Potentially invalid coordinates:", len(invalid_coordinates))

display(invalid_coordinates.head(10))

Potentially invalid coordinates: 42


,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
387565,18243,28.008978,-15.536867,bom retiro da esperanca,SP
513631,28165,41.614052,-8.411675,vila nova de campos,RJ
513643,28155,-34.586422,-58.732101,santa maria,RJ
513754,28155,42.439286,13.820214,santa maria,RJ
514429,28333,38.381672,-6.328200,raposo,RJ
516682,28595,43.684961,-7.411080,portela,RJ
538512,29654,29.409252,-98.484121,santo antônio do canaã,ES
538557,29654,21.657547,-101.466766,santo antonio do canaa,ES
585242,35179,25.995203,-98.078544,santana do paraíso,MG
585260,35179,25.995245,-98.078533,santana do paraiso,MG


In [39]:
# Add data quality flag for outlying coordinates
geoloc["invalid_coordinates"] = (
    ~geoloc["geolocation_lat"].between(-34, 6) |
    ~geoloc["geolocation_lng"].between(-74, -34)
)

In [6]:
# Check for any whitespace
string_columns = geoloc.select_dtypes(
    include=["object", "str"]
).columns

for column in string_columns:
    whitespace_count = geoloc[column].str.contains(
        r"^\s|\s$", na=False
    ).sum()

    print(f"{column}: {whitespace_count} values with whitespace")

for column in string_columns:
    print(
        f"{column}:",
        (geoloc[column] == "").sum(),
        "empty strings"
    )

geoloc.columns = (
    geoloc.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(geoloc.columns.tolist())

geolocation_city: 1 values with whitespace
geolocation_state: 0 values with whitespace
geolocation_city: 0 empty strings
geolocation_state: 0 empty strings
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']


In [8]:
whitespace_cities = geoloc[
    geoloc["geolocation_city"].str.contains(
        r"^\s|\s$", na=False
    )
]

display(whitespace_cities[["geolocation_city"]])

geoloc["geolocation_city"] = geoloc["geolocation_city"].str.strip()

,geolocation_city
670078,salvador


In [10]:
for column in string_columns:
    whitespace_count = geoloc[column].str.contains(
        r"^\s|\s$", na=False
    ).sum()

    print(f"{column}: {whitespace_count} values with whitespace")

geolocation_city: 0 values with whitespace
geolocation_state: 0 values with whitespace


In [42]:
geoloc.to_csv(
    "../data/processed/olist_geolocation_cleaned.csv",
    index=False
)